# 01 - Chunking Stats for Thesis\nNotebook này thống kê chất lượng chunk để dùng cho phần thực nghiệm của luận văn chunking.

In [ ]:
from pathlib import Path\nimport json, re, pandas as pd\n\nchunk_dir = Path('data/chunked')\nfiles = sorted(chunk_dir.glob('chunked_*.jsonl'))\nrows = []\nfor fp in files:\n    lines = [l.strip() for l in fp.read_text(encoding='utf-8').splitlines() if l.strip()]\n    lens=[]; markdown=0; legal=0; bad=0\n    for ln in lines:\n        try:\n            obj = json.loads(ln)\n            text = str(obj.get('text',''))\n            if not isinstance(obj, dict): bad += 1\n            lens.append(len(text))\n            if text.startswith('#') or '\n## ' in text or '\n### ' in text: markdown += 1\n            if re.search(r'(PHẦN|CHƯƠNG|MỤC|Điều|Khoản)', text): legal += 1\n        except Exception:\n            bad += 1\n    c=len(lines)\n    rows.append({\n        'file': fp.name,\n        'count': c,\n        'avg_len': round(sum(lens)/len(lens),1) if lens else 0,\n        'min_len': min(lens) if lens else 0,\n        'max_len': max(lens) if lens else 0,\n        'markdown_rate': round(markdown/c,3) if c else 0,\n        'legal_rate': round(legal/c,3) if c else 0,\n        'bad_lines': bad\n    })\n\ndf = pd.DataFrame(rows)\ndf

In [ ]:
out = Path('results/chunk_quality_summary.csv')\nout.parent.mkdir(parents=True, exist_ok=True)\ndf.to_csv(out, index=False, encoding='utf-8-sig')\nprint('Saved:', out.resolve())